# Baba Is AI: Backend Extraction Analysis
**Goal:** Investigate whether Baba Is AI exposes structured backend information that can be used as ground-truth labels for visual observations.

Because `baba-is-ai` is an external dependency, this notebook uses Python introspection to dynamically traverse the environment's memory, find the grid containers, and extract the rules without guessing API names.

```bash
python fmri_play.py --subject sub-01 --dummy-trigger --curriculum configs/dbp_games/baba__make_win.json
```

data `data/sub-01_20260910-155703`


### Load the Environment


In [ ]:
import baba
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# 1. Identify how Baba Is AI is instantiated and load a level
env = baba.make("env/make_win")
obs = env.reset()

print("Level successfully loaded!")



### Inspect the Environment
We recursively inspect `env.unwrapped` to see what attributes, grids, and rule containers are exposed to Python.

In [ ]:
print("=== Environment Hierarchy ===")
print("Env type:", type(env))
unwrapped = env.unwrapped if hasattr(env, "unwrapped") else env
print("Unwrapped type:", type(unwrapped))

print("\n=== Unwrapped Attributes ===")
for attr in dir(unwrapped):
    if not attr.startswith("__"):
        val = getattr(unwrapped, attr)
        print(f"{attr}: {type(val)}")


### Find Exposed Backend Information
Based on the hierarchy, we know the grid is at `env.unwrapped.grid` and rules are at `env.unwrapped._ruleset`. Let's inspect their internal structures!


In [ ]:
print("=== Inspecting the Grid ===")
grid_obj = unwrapped.grid
print("Grid type:", type(grid_obj))
print("Grid attributes:", vars(grid_obj))

print("\n=== Inspecting the Ruleset ===")
ruleset = unwrapped._ruleset
print("Ruleset type:", type(ruleset))
print("Ruleset attributes:", vars(ruleset))



### Produce Structured Table & Check Overlaps



In [ ]:
import json
structured_obs = []

grid_w = unwrapped.width
grid_h = unwrapped.height
grid = unwrapped.grid

# Attempt to extract the grid content
grid_array = None
if hasattr(grid, "grid"):
    grid_array = grid.grid
elif hasattr(grid, "board"):
    grid_array = grid.board

# Iterate over the grid to extract objects
if grid_array is not None:
    for idx, cell_content in enumerate(grid_array):
        if cell_content is None: continue
        
        # Calculate coordinates if it's a flat 1D array
        if isinstance(grid_array, list) and len(grid_array) == grid_w * grid_h:
            x = idx % grid_w
            y = idx // grid_w
        else:
            x, y = None, None # Fallback if it's not a standard 1D array
            
        # Baba Is You allows overlapping text and objects!
        items = cell_content if isinstance(cell_content, (list, tuple)) else [cell_content]
        
        for item in items:
            if item is None: continue
            
            # Extract standard fields
            name = getattr(item, "type", getattr(item, "name", str(item)))
            props = getattr(item, "properties", getattr(item, "state", []))
            
            # If the item itself has x,y coords, override
            x = getattr(item, "x", getattr(item, "pos", [x, None])[0])
            y = getattr(item, "y", getattr(item, "pos", [None, y])[1])
            
            structured_obs.append({
                "name": name,
                "x": x,
                "y": y,
                "properties": list(props) if isinstance(props, (set, list, tuple)) else props,
                "backend_type": str(type(item))
            })
else:
    print("Could not find the internal grid array. Please check the 'Inspecting the Grid' output!")

print("=== Structured Backend Observation (First 10) ===")
print(json.dumps(structured_obs[:10], indent=2))
print(f"... and {max(0, len(structured_obs) - 10)} more.\n")

# Check for overlaps (Goal 9)
coord_map = {}
has_overlaps = False
for obj in structured_obs:
    if obj["x"] is not None and obj["y"] is not None:
        pos = (obj["x"], obj["y"])
        if pos not in coord_map:
            coord_map[pos] = []
        coord_map[pos].append(obj["name"])
        if len(coord_map[pos]) > 1:
            has_overlaps = True

print("=== Cell Overlap Analysis ===")
if has_overlaps:
    print("YES! Multiple objects CAN occupy the same cell.")
    for pos, names in coord_map.items():
        if len(names) > 1:
            print(f"Cell {pos} contains: {names}")
else:
    print("No overlapping objects detected in this specific state, but the list structure implies it's possible.")



### Inspect Active Rules


In [ ]:
print("=== Active Rules Representation ===")
print("\n1. Default/Init Rules:")
print(getattr(unwrapped, "init_rules", "Not found"))

print("\n2. Current Ruleset:")
if hasattr(unwrapped, "_ruleset"):
    ruleset = unwrapped._ruleset
    if hasattr(ruleset, "rules"):
        print("Raw rules array:", ruleset.rules)
    else:
        print("Ruleset vars:", vars(ruleset))

print("\n3. Target Win Condition:")
print("Win Rule:", getattr(unwrapped, "win_rule", "Not found"))
print("Win Objects:", getattr(unwrapped, "win_obj_set", "Not found"))



### Find Missing Objects (BABA, WIN, etc.)
It looks like `grid.grid` only contains the static walls. Let's dynamically search the environment to find where the movable objects (BABA, texts) are stored!

In [ ]:
def find_all_objects(obj, current_path="env", depth=0, found=None, visited=None):
    if found is None: found = []
    if visited is None: visited = set()
    
    # Avoid infinite recursion (circular references)
    obj_id = id(obj)
    if obj_id in visited or depth > 6:
        return found
    visited.add(obj_id)
    
    # Is this a game object?
    is_wall = (getattr(obj, 'name', '') == 'wall' or getattr(obj, 'type', '') == 'wall')
    if hasattr(obj, 'name') or hasattr(obj, 'type'):
        if not is_wall:
            name = getattr(obj, 'name', getattr(obj, 'type', '?'))
            # Filter out generic things like environments, only get game entities
            if isinstance(name, str) and not name.startswith('<') and name != '?':
                x = getattr(obj, 'x', getattr(obj, 'pos', [None, None])[0])
                y = getattr(obj, 'y', getattr(obj, 'pos', [None, None])[1])
                found.append({"Name": name, "Path": current_path, "X": x, "Y": y})
            
    # Recurse into iterables
    if isinstance(obj, (list, tuple)):
        for i, item in enumerate(obj):
            find_all_objects(item, f"{current_path}[{i}]", depth+1, found, visited)
    elif hasattr(obj, '__dict__'):
        for k, v in vars(obj).items():
            if not k.startswith('__'):
                find_all_objects(v, f"{current_path}.{k}", depth+1, found, visited)
    elif isinstance(obj, dict):
        for k, v in obj.items():
            if isinstance(k, str) and not k.startswith('_'):
                find_all_objects(v, f"{current_path}['{k}']", depth+1, found, visited)
                
    return found

print("Searching for movable objects and texts...")
missing_objects = find_all_objects(unwrapped)

import pandas as pd
if missing_objects:
    df_objs = pd.DataFrame(missing_objects)
    df_objs = df_objs.drop_duplicates(subset=["Name", "X", "Y"])
    from IPython.display import display
    display(df_objs)
else:
    print("Could not find any movable objects.")


In [ ]:
rgb_frame = env.render("rgb_array") if hasattr(env, "render") else env.render()
if isinstance(rgb_frame, list): rgb_frame = rgb_frame[0]

fig, ax = plt.subplots(1, figsize=(10, 10))
ax.imshow(rgb_frame)

height, width, _ = rgb_frame.shape
cell_w = width / grid_w
cell_h = height / grid_h

print(f"Image dimensions: {width}x{height}")
print(f"Grid dimensions: {grid_w}x{grid_h}")
print(f"Calculated Cell Bounding Box Size: {cell_w}x{cell_h} pixels\n")

for obj in structured_obs:
    if obj["x"] is not None and obj["y"] is not None:
        x_px = obj["x"] * cell_w
        y_px = obj["y"] * cell_h
        
        rect = patches.Rectangle((x_px, y_px), cell_w, cell_h, linewidth=2, edgecolor='red', facecolor='none')
        ax.add_patch(rect)
        ax.text(x_px, y_px - 2, str(obj["name"]), color='red', fontsize=8, backgroundcolor='white')

plt.title("Backend Grid Objects Mapped Directly to Pixels")
plt.axis('off')
plt.show()



### Visual Scene Reproduction at t=10
Here we perfectly reproduce the exact pixel state at $t=10$ from your dataset by explicitly reseeding the engine and replaying your logged actions.

In [ ]:
%matplotlib inline
import os
import sys
import json
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

sys.path.append(os.path.abspath(".."))
from fmri_gym.adapters.baba import BabaAdapter

DATASET_DIR = "../data/sub-01_20260910-155703"
with open(os.path.join(DATASET_DIR, "manifest.json"), "r") as f:
    manifest = json.load(f)
    
game_curr = next(p for p in manifest["curriculum"] if p.get("type") == "game")
game_phase = next(p for p in manifest["phases"] if p.get("type") == "game")
data = np.load(os.path.join(DATASET_DIR, game_phase["data_file"]), allow_pickle=True)

actions = data["actions"]
ep_ids = data.get("episode_id", np.zeros(len(actions), dtype=int))
seeds = data["episode_seeds"]

TARGET_T = 10
target_ep = int(ep_ids[TARGET_T])
target_seed = int(seeds[target_ep])

print(f"Reconstructing t={TARGET_T} (which occurs in Episode {target_ep} with Seed {target_seed})...")

replay_env = BabaAdapter(game_curr)
current_ep = -1

for t in range(TARGET_T + 1):
    ep_id = int(ep_ids[t])
    if ep_id != current_ep:
        if current_ep != -1: 
            replay_env.close()
            replay_env = BabaAdapter(game_curr)
        replay_env.reset(seed=int(seeds[ep_id]))
        current_ep = ep_id
    
    if t < TARGET_T:
        replay_env.step(actions[t])

# Render the pixel array exactly at t=10
frame_t10 = replay_env.render()

fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(frame_t10)
ax.axis("off")
plt.title(f"Reconstructed Pixel Scene at t={TARGET_T}", fontsize=14, pad=15)
plt.show()
